# Project 25 — Multi-Hop RAG Research Agent
## Use Multiple Retrieval Hops Before Answering Complex Questions

**Stack:** LangGraph · Ollama · ChromaDB · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community langgraph chromadb

## Step 1 — Setup Knowledge Base

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.1)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

knowledge = [
    Document(page_content="The Transformer architecture was introduced by Vaswani et al. "
        "in 2017 in the paper 'Attention Is All You Need'. It uses self-attention "
        "instead of recurrence.", metadata={"topic": "transformer", "id": 1}),
    Document(page_content="BERT (Bidirectional Encoder Representations from Transformers) "
        "was created by Google in 2018. It uses masked language modeling and was "
        "pre-trained on BookCorpus and Wikipedia.", metadata={"topic": "bert", "id": 2}),
    Document(page_content="GPT-3 has 175 billion parameters and was trained by OpenAI. "
        "It uses autoregressive language modeling. Few-shot learning emerged as a "
        "key capability.", metadata={"topic": "gpt3", "id": 3}),
    Document(page_content="RAG was introduced by Lewis et al. in 2020. It combines "
        "a retriever (DPR) with a generator (BART). This reduces hallucination by "
        "grounding generation in retrieved documents.", metadata={"topic": "rag", "id": 4}),
    Document(page_content="DPR (Dense Passage Retrieval) uses dual BERT encoders — one "
        "for questions and one for passages. It outperforms BM25 on open-domain QA "
        "benchmarks.", metadata={"topic": "dpr", "id": 5}),
    Document(page_content="Self-attention computes query, key, value matrices from input. "
        "Attention scores = softmax(QK^T / sqrt(d)). Multi-head attention uses "
        "multiple parallel attention heads.", metadata={"topic": "attention", "id": 6}),
]

vectorstore = Chroma.from_documents(knowledge, embeddings, collection_name="multihop")
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print(f"Knowledge base: {len(knowledge)} documents")

## Step 2 — Define Multi-Hop Graph with LangGraph

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

class ResearchState(TypedDict):
    question: str
    sub_questions: list[str]
    retrieved_context: Annotated[list[str], operator.add]
    hop_count: int
    final_answer: str

def decompose_question(state: ResearchState) -> ResearchState:
    """Break complex question into sub-questions."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Break this complex question into 2-3 simpler sub-questions "
         "that need to be answered first. Return one question per line."),
        ("human", "{question}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"question": state["question"]})
    subs = [q.strip() for q in result.strip().split("\n") if q.strip()][:3]
    print(f"  Decomposed into {len(subs)} sub-questions")
    return {"sub_questions": subs, "hop_count": 0}

def retrieve_hop(state: ResearchState) -> ResearchState:
    """Retrieve documents for current sub-question."""
    hop = state["hop_count"]
    if hop < len(state["sub_questions"]):
        sub_q = state["sub_questions"][hop]
        docs = retriever.invoke(sub_q)
        context = [f"[Hop {hop+1}: {sub_q}]\n" + d.page_content for d in docs]
        print(f"  Hop {hop+1}: Retrieved {len(docs)} docs for: {sub_q[:50]}...")
        return {"retrieved_context": context, "hop_count": hop + 1}
    return {"hop_count": hop}

def should_continue(state: ResearchState) -> str:
    if state["hop_count"] < len(state["sub_questions"]):
        return "retrieve_more"
    return "synthesize"

def synthesize(state: ResearchState) -> ResearchState:
    """Combine all retrieved evidence into a final answer."""
    all_context = "\n\n".join(state["retrieved_context"])
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Synthesize a comprehensive answer using ALL the retrieved evidence. "
         "Cite which hop/source each piece of information comes from."),
        ("human", "Question: {question}\n\nEvidence:\n{context}\n\nSynthesized answer:")
    ])
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"question": state["question"], "context": all_context})
    print(f"  Synthesized from {len(state['retrieved_context'])} evidence pieces")
    return {"final_answer": answer}

# Build the graph
graph = StateGraph(ResearchState)
graph.add_node("decompose", decompose_question)
graph.add_node("retrieve", retrieve_hop)
graph.add_node("synthesize", synthesize)

graph.set_entry_point("decompose")
graph.add_edge("decompose", "retrieve")
graph.add_conditional_edges("retrieve", should_continue, {
    "retrieve_more": "retrieve",
    "synthesize": "synthesize",
})
graph.add_edge("synthesize", END)

app = graph.compile()
print("Multi-hop research graph compiled!")

## Step 3 — Run Multi-Hop Queries

In [ ]:
complex_questions = [
    "How does the attention mechanism used in BERT relate to the original Transformer?",
    "What retrieval method does RAG use, and how does it compare to keyword search?",
    "Trace the evolution from Transformers to GPT-3 — what are the key architectural changes?",
]

for q in complex_questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print("-"*60)
    result = app.invoke({"question": q, "sub_questions": [], "retrieved_context": [], "hop_count": 0, "final_answer": ""})
    print(f"\nFinal Answer:")
    print(result["final_answer"])
    print(f"\nTotal evidence pieces: {len(result['retrieved_context'])}")

## What You Learned
- **Question decomposition** for complex multi-part queries
- **Iterative retrieval** (multi-hop) using LangGraph
- **Evidence synthesis** across multiple retrieval rounds
- **Graph-based workflow** with conditional branching